In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Ramclass").getOrCreate()

In [0]:
df = spark.read.option("inferSchema",True).option("header",True).csv("/Volumes/workspace/default/sales/Landing_raw/customers_raw.csv")

In [0]:
df.show()
# df.printSchema()
#df.columns
df.dtypes


In [0]:
df.count()

In [0]:
df.describe().show()
df.select("customer_id","first_name","email").show(truncate=False)
df.select("customer_id","first_name","email")

In [0]:
from pyspark.sql.functions import col,lit,current_date,concat
df1 = df.select("*").where(col("signup_date") > lit('2025-01-01'))
df1.show()

In [0]:
df1.withColumn("Updated_date",lit(current_date())).show(truncate=False)

In [0]:
df1.withColumnRenamed("status","current_status").show(truncate=False)

In [0]:
df2 = df1.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name"))).drop("first_name","last_name")
df2.show()

In [0]:
df2.select("customer_id",col("full_name").alias("Name"),"email","country").show()

In [0]:
#Aggregation
dfx = spark.sql(f"""select * from employeeproject.default.employee""")
dfx.show()

In [0]:
from pyspark.sql.functions import col, sum, avg, count, max

dfx.groupBy("department") \
  .agg(
      sum("salary").alias("total_salary"),
      avg("salary").alias("avg_salary"),
      count("*").alias("employee_count"),
      max("salary").alias("top_salary")
  ).show()

In [0]:
%sql
select department,sum(salary) as total_salary,avg(salary) as avg_salary,count(*) as employee_count,max(salary) as top_salary
from employeeproject.default.employee
group by department

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, round as spark_round
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("TrainingDemo").getOrCreate()

# ============================================================
# SETUP: Create a single DataFrame with intentional duplicates
#        and nulls to demonstrate all operations
# ============================================================

data = [
    (1,  "Alice",   "Sales",    "2021-03-01", 75000,  12,  4200.50),
    (2,  "Bob",     "IT",       "2020-07-15", 90000,  8,   None),      # null commission
    (3,  "Carol",   "Sales",    "2019-11-20", 68000,  15,  5100.00),
    (4,  "David",   "HR",       None,         55000,  None, 0.0),      # null join_date, bonus_pct
    (5,  "Eve",     "Sales",    "2022-01-10", 82000,  10,  3800.75),
    (6,  "Frank",   None,       "2021-06-30", 61000,  5,   1500.00),   # null department
    (7,  "Alice",   "Sales",    "2021-03-01", 75000,  12,  4200.50),   # full duplicate of row 1
    (8,  "Bob",     "IT",       "2020-07-15", 90000,  8,   None),      # full duplicate of row 2
    (9,  "Grace",   "Finance",  "2018-05-22", 95000,  20,  6000.00),
    (10, "Hank",    "IT",       "2023-02-14", 72000,  None, None),     # multiple nulls
    (5,  "Eve",     "Sales",    "2022-01-10", 82000,  10,  3800.75),   # full duplicate of row 5
    (11, "Ivy",     "HR",       "2020-09-01", 58000,  6,   1200.00),
]

schema = StructType([
    StructField("emp_id",       IntegerType(), True),
    StructField("name",         StringType(),  True),
    StructField("department",   StringType(),  True),
    StructField("join_date",    StringType(),  True),
    StructField("salary",       IntegerType(), True),
    StructField("bonus_pct",    IntegerType(), True),
    StructField("commission",   DoubleType(),  True),
])

df = spark.createDataFrame(data, schema)

print("=== RAW DATA (12 rows including duplicates and nulls) ===")
df.show()



In [0]:
# ============================================================
# 1. .distinct() vs .dropDuplicates()
# ============================================================

print("=== 1a. distinct() — removes fully identical rows ===")
# distinct() checks ALL columns; goes from 12 → 9 rows
df_distinct = df.distinct()
df_distinct.show()
print(f"Row count after distinct(): {df_distinct.count()}")  # 11

print("=== 1b. dropDuplicates(['name','department']) — dedupe on specific columns ===")
# Keeps only the FIRST occurrence of each name+department combo
df_deduped = df.dropDuplicates(["name", "department"])
df_deduped.show()
print(f"Row count after dropDuplicates on name+dept: {df_deduped.count()}")


In [0]:
# ============================================================
# 2. .orderBy() / .sort()
# ============================================================

print("=== 2a. orderBy() — ascending by salary (default) ===")
df_distinct.orderBy("salary").show()

print("=== 2b. orderBy() — descending by salary, then name ascending ===")
df_distinct.orderBy(col("salary").desc(), col("name").asc()).show()

print("=== 2c. sort() — alias for orderBy(), same behaviour ===")
df_distinct.sort(col("commission").desc()).show()

# Key point: orderBy/sort are identical — sort is just a shorthand alias

In [0]:
# ============================================================
# 3. .limit() — preview large datasets
# ============================================================

print("=== 3. limit() — show only top 3 rows ===")
# Useful during development to avoid scanning full tables
df_distinct.orderBy(col("salary").desc()).limit(3).show()

# Tip: unlike .show(3), limit() returns a DataFrame you can chain further
top3_df = df_distinct.orderBy(col("salary").desc()).limit(3)
print(f"top3_df row count: {top3_df.count()}")  # 3

In [0]:
# ============================================================
# 4. Handling Nulls
# ============================================================
# where commission is null;
print("=== 4a. isNull() — filter rows WHERE commission IS NULL ===")
df_distinct.filter(col("commission").isNull()).show()

# where commission is not null;
print("=== 4b. isNotNull() — filter rows WHERE commission IS NOT NULL ===")
df_distinct.filter(col("commission").isNotNull()).show()

# Replacing Nulls
print("=== 4c. fillna() — fill nulls with default values ===")
# Pass a dict to fill different columns with different defaults
df_filled = df_distinct.fillna({
    "commission": 0.0,
    "bonus_pct":  0,
    "department": "Unknown",
    "join_date":  "1900-01-01"
})
df_filled.show()

print("=== 4d. dropna() — drop rows that have ANY null ===")
df_no_nulls = df_distinct.dropna()  # drops row if ANY column is null
df_no_nulls.show()
print(f"Rows after dropna(): {df_no_nulls.count()}")

print("=== 4e. dropna(subset) — drop rows where SPECIFIC columns are null ===")
# Only drop rows where bonus_pct or commission is null
df_drop_subset = df_distinct.dropna(subset=["bonus_pct", "commission"])
df_drop_subset.show()
print(f"Rows after dropna(subset): {df_drop_subset.count()}")

In [0]:
# ============================================================
# 5. .cast() — Type Casting
# ============================================================

print("=== 5. cast() — type conversions ===")

df_cast = df_filled \
    .withColumn("salary_double",    col("salary").cast(DoubleType())) \
    .withColumn("bonus_pct_str",    col("bonus_pct").cast(StringType())) \
    .withColumn("join_date_proper", col("join_date").cast("date"))       # string → date

df_cast.select("name", "salary", "salary_double", "bonus_pct", "bonus_pct_str", "join_date", "join_date_proper").show()
df_cast.printSchema()  # confirm new types


In [0]:
# ============================================================
# 6. Arithmetic Operations
# ============================================================

print("=== 6. Arithmetic Operations ===")

df_math = df_filled \
    .withColumn("annual_bonus",
                spark_round(col("salary") * (col("bonus_pct") / 100), 2)) \
    .withColumn("total_comp",
                spark_round(col("salary") + col("annual_bonus") + col("commission"), 2)) \
    .withColumn("daily_rate",
                spark_round(col("salary") / 260, 2)) \             # 260 working days
    .withColumn("commission_uplift_pct",
                spark_round((col("commission") / col("salary")) * 100, 2))

df_math.select(
    "name", "department", "salary",
    "bonus_pct", "annual_bonus",
    "commission", "total_comp",
    "daily_rate", "commission_uplift_pct"
).orderBy(col("total_comp").desc()).show()


CHEATSHEET
──────────────────────────────────────────────────────────────

distinct()                     → dedupe ALL columns
dropDuplicates(["col1","col2"])→ dedupe on SPECIFIC columns

orderBy("col")                 → ASC by default
orderBy(col("col").desc())     → DESC
sort(...)                      → alias for orderBy

limit(n)                       → returns a DataFrame with n rows

col("x").isNull()              → filter WHERE x IS NULL
col("x").isNotNull()           → filter WHERE x IS NOT NULL
fillna({"col": default})       → replace nulls with value
dropna()                       → drop rows with ANY null
dropna(subset=["c1","c2"])     → drop rows with null in c1/c2

col("x").cast("double")        → string shorthand
col("x").cast(DoubleType())    → explicit type class

col("a") + col("b")            → addition
col("a") * col("b") / 100      → multiply then divide
round(col("x"), 2)             → rounds to 2 decimal places
──────────────────────────────────────────────────────────────